# 00_runtime_mode_and_user_config

In [ ]:
from __future__ import annotations
import datetime as dt
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path
RUN_PROFILE = os.environ.get('TSTW_RUN_PROFILE', 'stage2_completion_formal').strip()
RUN_MODE = 'formal'
REQUIRE_FORMAL_PASS = True
DRIVE_ROOT = Path('/content/drive/MyDrive')
TSTW_ROOT = DRIVE_ROOT / 'TSTW'
MODELS_ROOT = DRIVE_ROOT / 'Models'
LOCAL_TSTW_ROOT = Path('/content/TSTW_runtime')
LOCAL_REPO_DIR = LOCAL_TSTW_ROOT / 'repo'
LOCAL_MODELS_DIR = LOCAL_TSTW_ROOT / 'models'
LOCAL_DATASET_CACHE_DIR = LOCAL_TSTW_ROOT / 'dataset_cache'
LOCAL_DATASET_DIR = LOCAL_TSTW_ROOT / 'datasets' / 'real_video_probe'
LOCAL_RUNS_DIR = LOCAL_TSTW_ROOT / 'runs'
LOCAL_TMP_DIR = LOCAL_TSTW_ROOT / 'tmp'
OVERRIDE_ROOT = TSTW_ROOT / 'configs' / RUN_PROFILE
DRIVE_STATE_PATH = TSTW_ROOT / 'registry' / 'drive_state.json'
RUNTIME_OVERRIDE_PATH = OVERRIDE_ROOT / 'runtime_override.json'
DATASET_OVERRIDE_PATH = OVERRIDE_ROOT / 'dataset_override.json'
MODEL_OVERRIDE_PATH = OVERRIDE_ROOT / 'model_override.json'
RESULTS_ROOT = TSTW_ROOT / 'results' / RUN_PROFILE
RESULT_REGISTRY_PATH = TSTW_ROOT / 'registry' / 'result_registry.jsonl'
REPO_URL = os.environ.get('TSTW_VW_REPO_URL', 'https://github.com/your-org/TSTW-VW.git').strip()
REPO_BRANCH = os.environ.get('TSTW_VW_REPO_BRANCH', 'main').strip()
HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if not REPO_URL:
    raise ValueError('TSTW_VW_REPO_URL is required')
print({'run_profile': RUN_PROFILE, 'run_mode': RUN_MODE, 'repo_url': REPO_URL, 'repo_branch': REPO_BRANCH, 'drive_root': str(DRIVE_ROOT), 'local_repo_dir': str(LOCAL_REPO_DIR), 'hf_token_present': bool(HF_TOKEN)})

# 01_mount_google_drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 02_read_drive_state_and_overrides

In [ ]:
if not DRIVE_STATE_PATH.exists():
    raise FileNotFoundError(DRIVE_STATE_PATH)
for override_path in (RUNTIME_OVERRIDE_PATH, DATASET_OVERRIDE_PATH, MODEL_OVERRIDE_PATH):
    if not override_path.exists():
        raise FileNotFoundError(override_path)
drive_state = json.loads(DRIVE_STATE_PATH.read_text(encoding='utf-8'))
runtime_override = json.loads(RUNTIME_OVERRIDE_PATH.read_text(encoding='utf-8'))
dataset_override = json.loads(DATASET_OVERRIDE_PATH.read_text(encoding='utf-8'))
model_override = json.loads(MODEL_OVERRIDE_PATH.read_text(encoding='utf-8'))
dataset_key = str(dataset_override.get('dataset_key') or drive_state.get('default_dataset_key') or '').strip()
if not dataset_key:
    raise ValueError('dataset_key is required in dataset_override or drive_state')
runtime_override['run_profile'] = RUN_PROFILE
runtime_override['repo_url'] = REPO_URL
runtime_override['repo_branch'] = REPO_BRANCH
runtime_override['hf_token_present'] = bool(HF_TOKEN)
print({'drive_state': str(DRIVE_STATE_PATH), 'runtime_override': str(RUNTIME_OVERRIDE_PATH), 'dataset_override': str(DATASET_OVERRIDE_PATH), 'model_override': str(MODEL_OVERRIDE_PATH), 'dataset_key': dataset_key})

# 03_prepare_local_workspace

In [ ]:
LOCAL_TSTW_ROOT.mkdir(parents=True, exist_ok=True)
for runtime_path in (LOCAL_MODELS_DIR, LOCAL_DATASET_CACHE_DIR, LOCAL_DATASET_DIR, LOCAL_RUNS_DIR, LOCAL_TMP_DIR):
    if runtime_path.exists():
        shutil.rmtree(runtime_path)
    runtime_path.mkdir(parents=True, exist_ok=True)
if LOCAL_REPO_DIR.exists():
    shutil.rmtree(LOCAL_REPO_DIR)
LOCAL_REPO_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
if not RESULT_REGISTRY_PATH.parent.exists():
    RESULT_REGISTRY_PATH.parent.mkdir(parents=True, exist_ok=True)
print({'local_root': str(LOCAL_TSTW_ROOT), 'local_repo_dir': str(LOCAL_REPO_DIR), 'results_root': str(RESULTS_ROOT)})

# 04_clone_or_update_repository

In [ ]:
if (LOCAL_REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(LOCAL_REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(LOCAL_REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(LOCAL_REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if any(LOCAL_REPO_DIR.iterdir()):
        shutil.rmtree(LOCAL_REPO_DIR)
        LOCAL_REPO_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(LOCAL_REPO_DIR)], check=True)
git_commit = subprocess.check_output(['git', '-C', str(LOCAL_REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
git_status_short = subprocess.check_output(['git', '-C', str(LOCAL_REPO_DIR), 'status', '--short'], text=True).strip()
short_commit = git_commit[:7]
print({'git_commit': git_commit, 'short_commit': short_commit, 'has_uncommitted_changes': bool(git_status_short)})

# 05_install_dependencies

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(LOCAL_REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pytest', 'huggingface_hub'], check=True)
pip_freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
print({'pip_freeze_lines': len([line for line in pip_freeze.splitlines() if line.strip()])})

# 06_copy_and_validate_dataset

In [ ]:
cache_tar_path = Path(str(dataset_override.get('cache_tar_path', '')).strip())
cache_tar_sha256 = str(dataset_override.get('cache_tar_sha256', '')).strip().lower()
if not cache_tar_path.exists():
    raise FileNotFoundError(cache_tar_path)
local_tar_path = LOCAL_DATASET_CACHE_DIR / cache_tar_path.name
shutil.copy2(cache_tar_path, local_tar_path)
if cache_tar_sha256:
    digest = hashlib.sha256(local_tar_path.read_bytes()).hexdigest().lower()
    if digest != cache_tar_sha256:
        raise RuntimeError({'expected_sha256': cache_tar_sha256, 'actual_sha256': digest, 'tar_path': str(local_tar_path)})
subprocess.run(['tar', '--zstd', '-xf', str(local_tar_path), '-C', str(LOCAL_DATASET_DIR)], check=True)
local_mp4_count = len(list(LOCAL_DATASET_DIR.rglob('*.mp4')))
if local_mp4_count < 1:
    raise RuntimeError('No mp4 files found after dataset extraction')
runtime_override['local_dataset_root'] = str(LOCAL_DATASET_DIR)
runtime_override['dataset_cache_tar_local_path'] = str(local_tar_path)
runtime_override['dataset_key'] = dataset_key
print({'dataset_tar': str(local_tar_path), 'mp4_count': local_mp4_count, 'local_dataset_root': str(LOCAL_DATASET_DIR)})

# 07_copy_and_validate_models

In [ ]:
def _copy_required_model(src_key: str, dst_subdir: str, required: bool = True) -> str:
    src_path = Path(str(model_override.get(src_key, '')).strip())
    if not src_path.exists():
        if required:
            raise FileNotFoundError({'missing_model_key': src_key, 'path': str(src_path)})
        return ''
    dst_path = LOCAL_MODELS_DIR / dst_subdir
    if dst_path.exists():
        shutil.rmtree(dst_path)
    shutil.copytree(src_path, dst_path)
    file_count = len([path for path in dst_path.rglob('*') if path.is_file()])
    if required and file_count < 1:
        raise RuntimeError({'empty_model_dir': str(dst_path), 'model_key': src_key})
    return str(dst_path)
local_vae_model_dir = _copy_required_model('vae_drive_model_root', 'vae/sd_vae_ft_mse', required=True)
local_lpips_model_dir = _copy_required_model('lpips_drive_model_root', 'lpips', required=True)
local_clip_model_dir = _copy_required_model('clip_drive_model_root', 'clip', required=False)
runtime_override['local_vae_model_dir'] = local_vae_model_dir
runtime_override['local_lpips_model_dir'] = local_lpips_model_dir
runtime_override['local_clip_model_dir'] = local_clip_model_dir or None
runtime_override['hf_token_present'] = bool(HF_TOKEN)
print({'local_vae_model_dir': local_vae_model_dir, 'local_lpips_model_dir': local_lpips_model_dir, 'local_clip_model_dir': local_clip_model_dir or None, 'hf_token_present': bool(HF_TOKEN)})

# 08_check_gpu_and_runtime

In [ ]:
gpu_name = ''
try:
    gpu_name = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
except Exception:
    gpu_name = ''
if RUN_MODE == 'formal' and not gpu_name:
    raise RuntimeError('formal mode requires GPU runtime in Colab')
runtime_override['python_version'] = sys.version.split()[0]
runtime_override['platform'] = platform.platform()
runtime_override['gpu_name'] = gpu_name or 'not_available'
print({'python': runtime_override['python_version'], 'platform': runtime_override['platform'], 'gpu': runtime_override['gpu_name']})

# 09_verify_repository_contract

In [ ]:
subprocess.run([sys.executable, 'tools/harness/validate_project_contract.py'], cwd=LOCAL_REPO_DIR, check=True)
audit_output = subprocess.check_output([sys.executable, 'tools/harness/run_all_audits.py'], cwd=LOCAL_REPO_DIR, text=True)
print(audit_output[-1000:] if len(audit_output) > 1000 else audit_output)

# 10_run_unit_tests_smoke

In [ ]:
pytest_output = subprocess.check_output([sys.executable, '-m', 'pytest', '-q', '-m', 'smoke', 'tests/test_real_video_vae_latent_records_schema.py', 'tests/test_real_video_vae_latent_table_rebuild.py', 'tests/test_real_video_vae_latent_drive_packager.py', 'tests/test_real_video_vae_latent_quality_metrics.py', 'tests/test_real_video_vae_latent_temporal_metrics.py'], cwd=LOCAL_REPO_DIR, text=True)
print(pytest_output)

# 11_run_stage2_completion_formal

In [ ]:
construction_phase = 'real_video_vae_latent_probe'
utc_time = dt.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
dataset_short = ''.join(character for character in dataset_key if character.isalnum() or character in ['_', '-'])[:32] or 'dataset'
run_id = f"{construction_phase}__{RUN_PROFILE}__{dataset_short}__{utc_time}__{short_commit}"
run_root = LOCAL_RUNS_DIR / run_id
runtime_config_path = run_root / 'artifacts' / 'colab_real_video_vae_latent_runtime_config.json'
run_root.mkdir(parents=True, exist_ok=True)
(run_root / 'artifacts').mkdir(parents=True, exist_ok=True)
runtime_override['run_id'] = run_id
runtime_override['git_commit'] = git_commit
runtime_override['dataset_manifest_snapshot_path'] = str(run_root / 'artifacts' / 'dataset_manifest_snapshot.json')
runtime_override['runtime_override_source'] = str(RUNTIME_OVERRIDE_PATH)
runtime_config_path.write_text(json.dumps(runtime_override, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
sys.path.insert(0, str(LOCAL_REPO_DIR))
from main.protocol.real_video_vae_latent_runner import RealVideoVaeLatentRunner
runner = RealVideoVaeLatentRunner(LOCAL_REPO_DIR)
formal_result = runner.run(output_root=run_root, run_mode='formal', runtime_profile_override='formal', runtime_config_path=runtime_config_path)
print({'run_id': formal_result.run_id, 'run_root': str(run_root), 'event_records': len(formal_result.event_score_records), 'threshold_records': len(formal_result.threshold_records)})

# 12_rebuild_tables_and_reports

In [ ]:
from main.analysis.real_video_vae_latent_artifacts import RealVideoVaeLatentArtifactBuilder
rebuilt_paths = RealVideoVaeLatentArtifactBuilder().rebuild_artifacts(run_root)
dataset_manifest_src = LOCAL_REPO_DIR / 'configs' / 'data' / 'real_video_probe_manifest.json'
dataset_manifest_dst = run_root / 'artifacts' / 'dataset_manifest_snapshot.json'
shutil.copy2(dataset_manifest_src, dataset_manifest_dst)
(run_root / 'artifacts' / 'config_snapshot').mkdir(parents=True, exist_ok=True)
for config_path in (RUNTIME_OVERRIDE_PATH, DATASET_OVERRIDE_PATH, MODEL_OVERRIDE_PATH):
    shutil.copy2(config_path, run_root / 'artifacts' / 'config_snapshot' / config_path.name)
print({key: str(value) for key, value in rebuilt_paths.items()})

# 13_validate_formal_outputs

In [ ]:
from main.colab.notebook_result_checker import check_real_video_vae_latent_outputs
formal_checks = check_real_video_vae_latent_outputs(run_root, run_mode='formal', require_formal_pass_criteria=REQUIRE_FORMAL_PASS)
checks_path_local = run_root / 'artifacts' / 'checks.json'
checks_path_local.write_text(json.dumps(formal_checks, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
if not formal_checks['status']:
    raise RuntimeError(formal_checks)
print(formal_checks)

# 14_pack_run_to_drive

In [ ]:
from main.colab.drive_packager import pack_real_video_vae_latent_run
compat_pack_root = run_root / 'artifacts' / 'compat_pack'
compat_pack_root.mkdir(parents=True, exist_ok=True)
compat_pack = pack_real_video_vae_latent_run(run_root=run_root, drive_output_dir=compat_pack_root, include_records=True, include_thresholds=True, include_tables=True, include_figures=True, include_reports=True, include_failure_gallery=True, include_manifest=True)
drive_archive_path = RESULTS_ROOT / f'{run_id}.tar.zst'
drive_summary_path = RESULTS_ROOT / f'{run_id}_summary.json'
drive_checks_path = RESULTS_ROOT / f'{run_id}_checks.json'
subprocess.run(['tar', '--zstd', '-cf', str(drive_archive_path), '-C', str(LOCAL_RUNS_DIR), run_id], check=True)
summary_payload = {'run_id': run_id, 'run_profile': RUN_PROFILE, 'construction_phase': construction_phase, 'dataset_key': dataset_key, 'git_commit': git_commit, 'decision': formal_checks.get('RealVideoVaeLatentDecision'), 'status': formal_checks.get('status'), 'archive_path': str(drive_archive_path), 'created_at': dt.datetime.utcnow().isoformat() + 'Z'}
drive_summary_path.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
drive_checks_path.write_text(json.dumps(formal_checks, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print({'archive_path': str(drive_archive_path), 'summary_path': str(drive_summary_path), 'checks_path': str(drive_checks_path), 'compat_zip_path': str(compat_pack['zip_path'])})

# 15_update_result_registry

In [ ]:
registry_entry = {'schema_version': 'tstw_result_registry_entry.v1', 'run_id': run_id, 'run_profile': RUN_PROFILE, 'construction_phase': construction_phase, 'dataset_key': dataset_key, 'git_commit': git_commit, 'archive_path': str(drive_archive_path), 'summary_path': str(drive_summary_path), 'checks_path': str(drive_checks_path), 'decision': formal_checks.get('RealVideoVaeLatentDecision'), 'created_at': dt.datetime.utcnow().isoformat() + 'Z'}
with RESULT_REGISTRY_PATH.open('a', encoding='utf-8') as handle:
    handle.write(json.dumps(registry_entry, ensure_ascii=False) + '\n')
print(registry_entry)

# 16_print_final_summary

In [ ]:
log_dir = run_root / 'logs'
log_dir.mkdir(parents=True, exist_ok=True)
(log_dir / 'colab.log').write_text('Notebook completed successfully.\n', encoding='utf-8')
(log_dir / 'pytest.log').write_text(pytest_output, encoding='utf-8')
(log_dir / 'audit.log').write_text(audit_output, encoding='utf-8')
(log_dir / 'dependency_freeze.txt').write_text(pip_freeze, encoding='utf-8')
(log_dir / 'git_commit.txt').write_text(git_commit + '\n', encoding='utf-8')
(log_dir / 'git_status.txt').write_text(git_status_short + '\n', encoding='utf-8')
summary = {'run_id': run_id, 'run_root': str(run_root), 'decision': formal_checks.get('RealVideoVaeLatentDecision'), 'next_allowed_stage': formal_checks.get('NextAllowedStage'), 'drive_archive_path': str(drive_archive_path), 'drive_summary_path': str(drive_summary_path), 'drive_checks_path': str(drive_checks_path), 'result_registry_path': str(RESULT_REGISTRY_PATH)}
(run_root / 'reports' / 'colab_final_summary.md').write_text('\n'.join([f'- {key}: {value}' for key, value in summary.items()]) + '\n', encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))